# Comparing GPT 5.2 and Gemini 3.0 with SAE Features

This notebook compares GPT 5.2 and Gemini 3.0 responses to 2000 Chatbot Arena prompts using SAE features.

**Feature frequency diffing** - Which features appear more often in one model's responses?

In other words, what are the overall qualitative differences between the models?

## Setup

In [ ]:
# !pip install git+https://github.com/nickjiang2378/interp_embed

In [2]:
from interp_embed import Dataset
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

In [3]:
def diff_features(
    ds1,
    ds2,
):
    """
    Calculate the differences in feature frequency (i.e. how often a feature is activated) between two datasets.
    """
    # Get per-feature activations (frequency of nonzero activation for each feature)
    fa1 = ds1.latents("binarize")  # shape: (N documents, F features)
    fa2 = ds2.latents("binarize")  # shape: (N documents, F features)

    freq1 = np.sum(fa1, axis=0) / fa1.shape[0]
    freq2 = np.sum(fa2, axis=0) / fa2.shape[0]

    diff = freq1 - freq2

    # Get feature labels
    feature_labels_dict = ds1.feature_labels()
    num_features = freq1.shape[0]
    feature_labels = [feature_labels_dict.get(i, "") for i in range(num_features)]

    df = pd.DataFrame({
        "feature": feature_labels,
        "feature_id": np.arange(num_features),
        ds1.id: freq1,
        ds2.id: freq2,
        "frequency_difference": diff,
    })

    return df


## Load Datasets

We load three precomputed `Dataset` objects, each containing SAE feature activations from 1998 randomly selected conversations from Chatbot Arena:
- **prompts**: Activations on the user prompts (shared across both models)
- **gpt**: Activations on GPT 5.2 responses
- **gemini**: Activations on Gemini 3.0 responses

We used Goodfire's LLama-3.1-8B-Instruct SAEs to embed these datasets.

In [4]:
# Define the model repo and artifact names/paths
repo_id = "nickjiang/feature_labels"

prompts_path = hf_hub_download(
        repo_id=repo_id,
        filename=f"artifacts/gpt52_gemini3_tutorial/prompts.pkl"
    )

gpt52_path = hf_hub_download(
    repo_id=repo_id,
    filename=f"artifacts/gpt52_gemini3_tutorial/gpt-52.pkl"
)

gemini3_path = hf_hub_download(
    repo_id=repo_id,
    filename=f"artifacts/gpt52_gemini3_tutorial/gemini-3.pkl"
)

prompts = Dataset.load_from_file(prompts_path)
gpt = Dataset.load_from_file(gpt52_path)
gemini = Dataset.load_from_file(gemini3_path)

print(f"GPT 5.2: {len(gpt)} documents")
print(f"Gemini 3: {len(gemini)} documents")
print(f"Prompts: {len(prompts)} documents")

artifacts/gpt52_gemini3_tutorial/gpt-52.(…):   0%|          | 0.00/1.22G [00:00<?, ?B/s]

artifacts/gpt52_gemini3_tutorial/gemini-(…):   0%|          | 0.00/1.37G [00:00<?, ?B/s]

GPT 5.2: 1998 documents
Gemini 3: 1998 documents
Prompts: 1998 documents


To see the original dataset of prompts and model responses, use the `.pandas()` method.

In [5]:
df = gemini.pandas()
df

,prompt,prompt_id,openai/gpt-5.2,google/gemini-3-pro-preview
0,in vulkan when recreating the swapchain. do i ...,0,You *can* reuse command buffers across swapcha...,The short answer is: **You generally cannot re...
1,What relative difficulty of the next games in ...,1,Here’s a **relative difficulty score (0–100)**...,Assessing video game difficulty is highly subj...
2,I am someone who wants to do data science. I ...,2,### JupyterLab vs VS Code for data science (Py...,This is a classic debate in the data science c...
3,What is special about Harvard classes of 1902 ...,3,Two Harvard graduating classes are especially ...,NaN
4,as a new employee what should i say on day 1 f...,4,"Keep it short, friendly, and focused on three ...",Congratulations on the new job! The first day ...
...,...,...,...,...
1993,Explain a race condition in one sentence,1995,A race condition is a bug where a program’s ou...,A race condition occurs when a program's behav...
1994,Tell me step by step if UAP’s could be real an...,1996,1) Define what “UAP” means \n- UAP = “Unident...,To determine if Unidentified Anomalous Phenome...
1995,Write a poem about code and music,1997,"In the hush of midnight monitors glow, \na st...","The cursor blinks, a silent beat,\nUpon the da..."
1996,Why should an analysis of a whodunnit account ...,1998,An analysis of a whodunnit should account for ...,An analysis of a whodunnit must account for th...


## Feature Frequency Diffing

For each SAE feature, we compute how frequently it activates across all documents in each model's responses. Then, we subtract the feature frequencies and find the features with the highest magnitude difference. Positive frequency difference = more frequent in GPT 5.2, negative feature difference = more frequent in Gemini 3.

In [6]:
freq_diff = diff_features(gpt, gemini)
freq_diff = freq_diff.sort_values(by="frequency_difference", ascending=False, key=abs).reset_index(drop=True)
freq_diff.head(20)

,feature,feature_id,gpt-5.2,gemini-3-pro-preview,frequency_difference
0,The assistant is about to demonstrate or show ...,24183,0.118118,0.680681,-0.562563
1,The assistant is about to provide a numbered l...,2805,0.146647,0.663163,-0.516517
2,Offensive request from the user,61439,0.114114,0.623624,-0.509510
3,"Toasts and dedications in text, particularly '...",7398,0.159159,0.665666,-0.506507
4,The assistant is about to present requested in...,20863,0.087087,0.569069,-0.481982
5,Middle items (particularly 2-9) in numbered lists,27340,0.218719,0.696697,-0.477978
6,Form field labels and section headers in struc...,55920,0.247247,0.713213,-0.465966
7,The assistant needs clarification and is polit...,62738,0.548048,0.086587,0.461461
8,Copular 'is' in declarative statements,36135,0.259760,0.688689,-0.428929
9,The assistant confirming they can help with a ...,9642,0.497497,0.069570,0.427928


We can find example prompts where a feature activates strongly in one model's response but not the other. This helps us understand what the frequency differences look like in practice. Consider feature #2805 ("numbered lists"), which appears more often in Gemini than GPT.

In [7]:
feature = 2805  # "The assistant is about to provide a numbered list"

latents1 = gemini.latents()
latents2 = gpt.latents()
df = gpt.pandas()

# Find documents where the feature activates in Gemini but not GPT
diff = latents1[:, feature] - latents2[:, feature]
diff[(diff < 0) | (latents2[:, feature] != 0)] = 0
top_document_index = np.argsort(diff)[-1]

print("===== Prompt =====")
print(df.iloc[top_document_index]["prompt"])
print("-" * 80)
print("===== Gemini =====")
print(df.iloc[top_document_index]["google/gemini-3-pro-preview"][:1000] + "...")
print("-" * 80)
print("===== GPT =====")
print(df.iloc[top_document_index]["openai/gpt-5.2"][:1000] + "...")

===== Prompt =====
Is rice bad for you
--------------------------------------------------------------------------------
===== Gemini =====
The short answer is **no, rice is not inherently bad for you.** It is a staple food for more than half of the world's population and provides a vital source of energy.

However, whether rice is "healthy" or "unhealthy" depends on the **type of rice** you eat, **how much** you eat, and your overall **dietary needs**.

Here is a breakdown of the pros and cons to help you decide.

### 1. White Rice vs. Brown Rice
The biggest factor is how the rice is processed.

*   **White Rice (Refined):**
    *   **The Process:** The husk, bran, and germ are removed. This removes much of the fiber and nutrients, leaving mostly starch.
    *   **The Downside:** It has a high Glycemic Index (GI), meaning it can cause rapid spikes in blood sugar. It is less filling than brown rice.
    *   **The Upside:** It is very easy to digest (great for people with stomach issues)